In [1]:
import pandas as pd
from sqlalchemy import text

# Utils moduli
from utils.data_loader import load_all_orders, load_returns, load_users
from utils.data_cleaner import clean_orders
from utils.db_connector import get_engine, create_database, test_connection
from utils.Exporter import export_csv

# Putanje do foldera
RAW_DIR       = 'data/raw/'
PROCESSED_DIR = 'data/processed/'

print("✅ Import uspešan!")

✅ Import uspešan!


In [2]:
create_database ('smart_store')

engine = get_engine('smart_store')

test_connection(engine)

✅ Baza 'smart_store' kreirana (ili već postoji)
✅ Konekcija kreirana: localhost/smart_store
✅ Konekcija uspešna!


True

In [3]:
orders_raw = load_all_orders(RAW_DIR)

returns_raw = load_returns(RAW_DIR + 'Returns.csv')
users_raw = load_users(RAW_DIR + 'Users.csv')

✅ Učitan Orders_East.csv: 474 redova
✅ Učitan Orders_West.csv: 470 redova
✅ Učitan Orders_South.csv: 441 redova
✅ Učitan Orders_Central.csv: 566 redova

📦 Ukupno redova nakon spajanja: 1951
✅ Učitan Returns: 300 redova
✅ Učitan Users: 4 redova


In [4]:
orders_raw.head()

,Row ID,Order Priority,Discount,Unit Price,Shipping Cost,Customer ID,Customer Name,Ship Mode,Customer Segment,Product Category,...,Region,State or Province,City,Postal Code,Order Date,Ship Date,Profit,Quantity ordered new,Sales,Order ID
0,21776,Critical,0.06,9.48,7.29,11,Marcus Dunlap,Regular Air,Home Office,Furniture,...,East,New Jersey,Roselle,7203,2015-02-15,2015-02-17,-53.8096,22,211.15,90192
1,18181,Critical,0.00,4.42,4.99,15,Timothy Reese,Regular Air,Small Business,Office Supplies,...,East,New York,Smithtown,11787,2015-04-08,2015-04-09,-59.8200,7,33.47,86837
2,20925,Medium,0.01,35.94,6.66,15,Timothy Reese,Regular Air,Small Business,Office Supplies,...,East,New York,Smithtown,11787,2015-05-28,2015-05-28,261.8757,10,379.53,86839
3,26267,High,0.04,2.98,1.58,16,Sarah Ramsey,Regular Air,Small Business,Office Supplies,...,East,New York,Syracuse,13210,2015-02-12,2015-02-15,2.6300,6,18.80,86836
4,26268,High,0.05,115.99,2.50,16,Sarah Ramsey,Regular Air,Small Business,Technology,...,East,New York,Syracuse,13210,2015-02-12,2015-02-14,652.7331,10,945.99,86836


In [5]:
orders_clean = clean_orders(orders_raw, returns_raw)

🔄 Počinje čišćenje podataka...

TRIM primenjen na sve string kolone
Postal Code formatiran na 5 cifara
Zaokružene kolone: ['Discount', 'Unit Price', 'Shipping Cost', 'Product Base Margin', 'Profit', 'Sales']
Uklonjeno duplikata: 0 (bilo 1951, ostalo 1951)
✅ Datumi konvertovani u datetime format
Dodata kolona: Days Between Order and Ship Date
Dodata kolona: is_returned (435 vraćenih narudžbina)
⚠️  Pronađene null vrednosti:
   - Product Base Margin: 16 null vrednosti
   ✅ Product Base Margin: null zamenjeni medijanom (0.53)

📊 Ukupno redova sa null vrednostima: 16

✅ Čišćenje završeno! Finalni DataFrame: 1951 redova, 27 kolona


In [6]:
orders_clean.head()

,Row ID,Order Priority,Discount,Unit Price,Shipping Cost,Customer ID,Customer Name,Ship Mode,Customer Segment,Product Category,...,City,Postal Code,Order Date,Ship Date,Profit,Quantity ordered new,Sales,Order ID,Days Between Order and Ship Date,is_returned
0,21776,Critical,0.06,9.48,7.29,11,Marcus Dunlap,Regular Air,Home Office,Furniture,...,Roselle,07203,2015-02-15,2015-02-17,-53.81,22,211.15,90192,2,False
1,18181,Critical,0.00,4.42,4.99,15,Timothy Reese,Regular Air,Small Business,Office Supplies,...,Smithtown,11787,2015-04-08,2015-04-09,-59.82,7,33.47,86837,1,False
2,20925,Medium,0.01,35.94,6.66,15,Timothy Reese,Regular Air,Small Business,Office Supplies,...,Smithtown,11787,2015-05-28,2015-05-28,261.88,10,379.53,86839,0,False
3,26267,High,0.04,2.98,1.58,16,Sarah Ramsey,Regular Air,Small Business,Office Supplies,...,Syracuse,13210,2015-02-12,2015-02-15,2.63,6,18.80,86836,3,False
4,26268,High,0.05,115.99,2.50,16,Sarah Ramsey,Regular Air,Small Business,Technology,...,Syracuse,13210,2015-02-12,2015-02-14,652.73,10,945.99,86836,2,False


In [7]:
export_csv(orders_clean, PROCESSED_DIR + 'orders_clean.csv')
export_csv(returns_raw,  PROCESSED_DIR + 'returns_clean.csv')
export_csv(users_raw,    PROCESSED_DIR + 'users_clean.csv')

✅ Exportovano: data/processed/orders_clean.csv (1951 redova)
✅ Exportovano: data/processed/returns_clean.csv (300 redova)
✅ Exportovano: data/processed/users_clean.csv (4 redova)


In [8]:
# Upload orders u staging tabelu
print("📤 Upload staging_orders...")
orders_clean.to_sql(
    name='staging_orders',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=500
)
print(f"✅ staging_orders uploadovan: {len(orders_clean)} redova")

# Upload returns
print("\n📤 Upload staging_returns...")
returns_raw.to_sql(
    name='staging_returns',
    con=engine,
    if_exists='replace',
    index=False
)
print(f"✅ staging_returns uploadovan: {len(returns_raw)} redova")

# Upload users
print("\n📤 Upload staging_users...")
users_raw.to_sql(
    name='staging_users',
    con=engine,
    if_exists='replace',
    index=False
)
print(f"✅ staging_users uploadovan: {len(users_raw)} redova")

📤 Upload staging_orders...
✅ staging_orders uploadovan: 1951 redova

📤 Upload staging_returns...
✅ staging_returns uploadovan: 300 redova

📤 Upload staging_users...
✅ staging_users uploadovan: 4 redova


In [9]:
with engine.connect() as conn:

    # Dim_Calendar
    print("📅 Punjenje Dim_Calendar...")
    conn.execute(text("""
        INSERT IGNORE INTO dim_calendar (date_id, full_date, year, quarter,
                                         month_number, month_name, week_number)
        SELECT DISTINCT
            CAST(DATE_FORMAT(`Order Date`, '%Y%m%d') AS UNSIGNED),
            DATE(`Order Date`),
            YEAR(`Order Date`),
            QUARTER(`Order Date`),
            MONTH(`Order Date`),
            MONTHNAME(`Order Date`),
            WEEK(`Order Date`)
        FROM staging_orders
        UNION
        SELECT DISTINCT
            CAST(DATE_FORMAT(`Ship Date`, '%Y%m%d') AS UNSIGNED),
            DATE(`Ship Date`),
            YEAR(`Ship Date`),
            QUARTER(`Ship Date`),
            MONTH(`Ship Date`),
            MONTHNAME(`Ship Date`),
            WEEK(`Ship Date`)
        FROM staging_orders;
    """))
    conn.commit()
    print("✅ Dim_Calendar napunjena")

    # Dim_Customers
    print("\n👤 Punjenje Dim_Customers...")
    conn.execute(text("""
        INSERT IGNORE INTO dim_customers (customer_id, customer_name)
        SELECT DISTINCT `Customer ID`, `Customer Name`
        FROM staging_orders;
    """))
    conn.commit()
    print("✅ Dim_Customers napunjena")

    # Dim_Product
    print("\n📦 Punjenje Dim_Product...")
    conn.execute(text("""
        INSERT IGNORE INTO dim_product
            (product_name, product_category, product_sub_category,
             product_container, product_base_margin)
        SELECT
            `Product Name`,
            `Product Category`,
            `Product Sub-Category`,
            `Product Container`,
            ROUND(AVG(`Product Base Margin`), 2)
        FROM staging_orders
        GROUP BY `Product Name`, `Product Category`,
                 `Product Sub-Category`, `Product Container`;
    """))
    conn.commit()
    print("✅ Dim_Product napunjena")

    # Dim_Geography
    print("\n🗺️  Punjenje Dim_Geography...")
    conn.execute(text("""
        INSERT IGNORE INTO dim_geography
            (country, region, state_or_province, city, postal_code)
        SELECT DISTINCT
            Country, Region, `State or Province`, City, `Postal Code`
        FROM staging_orders;
    """))
    conn.commit()
    print("✅ Dim_Geography napunjena")

    # Dim_Segment
    print("\n🏷️  Punjenje Dim_Segment...")
    conn.execute(text("""
        INSERT IGNORE INTO dim_segment (segment_name)
        SELECT DISTINCT `Customer Segment`
        FROM staging_orders;
    """))
    conn.commit()
    print("✅ Dim_Segment napunjena")

    # Dim_ShipMode
    print("\n🚚 Punjenje Dim_ShipMode...")
    conn.execute(text("""
        INSERT IGNORE INTO dim_shipmode (ship_mode)
        SELECT DISTINCT `Ship Mode`
        FROM staging_orders;
    """))
    conn.commit()
    print("✅ Dim_ShipMode napunjena")

    # Dim_OrderPriority
    print("\n⚡ Punjenje Dim_OrderPriority...")
    conn.execute(text("""
        INSERT IGNORE INTO dim_orderpriority (order_priority)
        SELECT DISTINCT `Order Priority`
        FROM staging_orders;
    """))
    conn.commit()
    print("✅ Dim_OrderPriority napunjena")

    # Dim_Manager
    print("\n👔 Punjenje Dim_Manager...")
    conn.execute(text("""
        INSERT IGNORE INTO dim_manager (manager_name, geography_id)
        SELECT DISTINCT
            u.Manager,
            g.geography_id
        FROM staging_users u
        JOIN dim_geography g ON g.region = u.Region
        GROUP BY u.Manager, g.geography_id
        LIMIT 4;
    """))
    conn.commit()
    print("✅ Dim_Manager napunjena")

📅 Punjenje Dim_Calendar...
✅ Dim_Calendar napunjena

👤 Punjenje Dim_Customers...
✅ Dim_Customers napunjena

📦 Punjenje Dim_Product...
✅ Dim_Product napunjena

🗺️  Punjenje Dim_Geography...
✅ Dim_Geography napunjena

🏷️  Punjenje Dim_Segment...
✅ Dim_Segment napunjena

🚚 Punjenje Dim_ShipMode...
✅ Dim_ShipMode napunjena

⚡ Punjenje Dim_OrderPriority...
✅ Dim_OrderPriority napunjena

👔 Punjenje Dim_Manager...
✅ Dim_Manager napunjena


In [11]:
# Provera da li sve dimenzije imaju podatke
with engine.connect() as conn:
    tables = [
        'dim_customers', 'dim_product', 'dim_geography',
        'dim_segment', 'dim_orderpriority', 'dim_shipmode',
        'dim_manager', 'dim_calendar'
    ]
    
    print("📊 Broj redova po dimenziji:")
    print("-" * 35)
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"  {table:<25} {count:>6} redova")

📊 Broj redova po dimenziji:
-----------------------------------
  dim_customers               1130 redova
  dim_product                  912 redova
  dim_geography                985 redova
  dim_segment                    4 redova
  dim_orderpriority              5 redova
  dim_shipmode                   3 redova
  dim_manager                    4 redova
  dim_calendar                 188 redova


In [12]:
# Testiraj svaki JOIN posebno da nađemo problem
with engine.connect() as conn:

    # Test 1 - Customer JOIN
    result = conn.execute(text("""
        SELECT COUNT(*) FROM staging_orders s
        JOIN dim_customers c ON c.customer_id = s.`Customer ID`
    """))
    print(f"✅ Customer JOIN: {result.scalar()} redova")

    # Test 2 - Product JOIN
    result = conn.execute(text("""
        SELECT COUNT(*) FROM staging_orders s
        JOIN dim_product p ON p.product_name = s.`Product Name`
    """))
    print(f"✅ Product JOIN: {result.scalar()} redova")

    # Test 3 - Geography JOIN
    result = conn.execute(text("""
        SELECT COUNT(*) FROM staging_orders s
        JOIN dim_geography g ON g.state_or_province = s.`State or Province`
                             AND g.city             = s.City
                             AND g.postal_code      = s.`Postal Code`
    """))
    print(f"✅ Geography JOIN: {result.scalar()} redova")

    # Test 4 - Manager JOIN
    result = conn.execute(text("""
        SELECT COUNT(*) FROM staging_orders s
        JOIN dim_manager m ON m.manager_name IN (
            SELECT Manager FROM staging_users
            WHERE Region = s.Region
        )
    """))
    print(f"✅ Manager JOIN: {result.scalar()} redova")

    # Test 5 - Calendar JOIN
    result = conn.execute(text("""
        SELECT COUNT(*) FROM staging_orders s
        JOIN dim_calendar cal 
            ON cal.date_id = CAST(DATE_FORMAT(s.`Order Date`, '%Y%m%d') AS UNSIGNED)
    """))
    print(f"✅ Calendar JOIN: {result.scalar()} redova")

✅ Customer JOIN: 1951 redova
✅ Product JOIN: 1951 redova
✅ Geography JOIN: 1951 redova
✅ Manager JOIN: 1896 redova
✅ Calendar JOIN: 1951 redova


In [15]:
with engine.connect() as conn:
    conn.execute(text("DELETE FROM dim_manager;"))
    conn.commit()
    print("✅ Dim_Manager očišćena")

✅ Dim_Manager očišćena


In [16]:
with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO dim_manager (manager_name, geography_id)
        SELECT 
            u.Manager,
            MIN(g.geography_id)
        FROM staging_users u
        JOIN dim_geography g ON g.region = u.Region
        GROUP BY u.Manager;
    """))
    conn.commit()
    print("✅ Dim_Manager napunjena ispravno")

✅ Dim_Manager napunjena ispravno


In [18]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT m.manager_id, m.manager_name, g.region
        FROM dim_manager m
        JOIN dim_geography g ON g.geography_id = m.geography_id
    """))
    
    for row in result.fetchall():
        print(f"  ID: {row[0]}, Manager: {row[1]}, Region: {row[2]}")


  ID: 8, Manager: Erin, Region: East
  ID: 9, Manager: William, Region: West
  ID: 10, Manager: Sam, Region: South
  ID: 11, Manager: Chris, Region: Central


In [21]:
with engine.connect() as conn:
    print("📊 Punjenje Fact_Orders...")
    conn.execute(text("""
        INSERT INTO fact_orders (
            row_id, customer_id, product_id, geography_id,
            order_date_id, ship_date_id, segment_id,
            orderpriority_id, manager_id, shipmode_id,
            order_id, sales, profit, quantity,
            discount, unit_price, shipping_cost, is_returned
        )
        SELECT
            s.`Row ID`,
            c.customer_key,
            p.product_key,
            g.geography_id,
            CAST(DATE_FORMAT(s.`Order Date`, '%Y%m%d') AS UNSIGNED),
            CAST(DATE_FORMAT(s.`Ship Date`,  '%Y%m%d') AS UNSIGNED),
            seg.segment_id,
            op.orderpriority_id,
            m.manager_id,
            sm.shipmode_id,
            s.`Order ID`,
            s.Sales,
            s.Profit,
            s.`Quantity ordered new`,
            s.Discount,
            s.`Unit Price`,
            s.`Shipping Cost`,
            s.is_returned
        FROM staging_orders s
        JOIN dim_customers     c   ON c.customer_id       = s.`Customer ID`
        JOIN dim_product       p   ON p.product_name      = s.`Product Name`
        JOIN dim_geography     g   ON g.state_or_province = s.`State or Province`
                                  AND g.city              = s.City
                                  AND g.postal_code       = s.`Postal Code`
        JOIN dim_segment       seg ON seg.segment_name    = s.`Customer Segment`
        JOIN dim_orderpriority op  ON op.order_priority   = s.`Order Priority`
        JOIN dim_shipmode      sm  ON sm.ship_mode        = s.`Ship Mode`
        JOIN dim_manager       m   ON m.manager_name = (
            SELECT Manager FROM staging_users
            WHERE Region = s.Region
            LIMIT 1
        );
    """))
    conn.commit()
    print("✅ Fact_Orders napunjena!")

📊 Punjenje Fact_Orders...


IntegrityError: (pymysql.err.IntegrityError) (1452, 'Cannot add or update a child row: a foreign key constraint fails (`smart_store`.`fact_orders`, CONSTRAINT `fact_orders_ibfk_1` FOREIGN KEY (`customer_id`) REFERENCES `dim_customers` (`customer_id`))')
[SQL: 
        INSERT INTO fact_orders (
            row_id, customer_id, product_id, geography_id,
            order_date_id, ship_date_id, segment_id,
            orderpriority_id, manager_id, shipmode_id,
            order_id, sales, profit, quantity,
            discount, unit_price, shipping_cost, is_returned
        )
        SELECT
            s.`Row ID`,
            c.customer_key,
            p.product_key,
            g.geography_id,
            CAST(DATE_FORMAT(s.`Order Date`, '%%Y%%m%%d') AS UNSIGNED),
            CAST(DATE_FORMAT(s.`Ship Date`,  '%%Y%%m%%d') AS UNSIGNED),
            seg.segment_id,
            op.orderpriority_id,
            m.manager_id,
            sm.shipmode_id,
            s.`Order ID`,
            s

In [22]:
# Pronađi tačan uzrok greške
with engine.connect() as conn:
    
    # Proveri da li ima Row ID duplikata u staging
    result = conn.execute(text("""
        SELECT `Row ID`, COUNT(*) as cnt
        FROM staging_orders
        GROUP BY `Row ID`
        HAVING cnt > 1
    """))
    rows = result.fetchall()
    print(f"Duplikat Row ID u staging: {len(rows)}")
    for row in rows:
        print(f"  Row ID: {row[0]}, pojavljuje se: {row[1]} puta")
    
    # Proveri da li fact_orders vec ima podatke
    result = conn.execute(text("SELECT COUNT(*) FROM fact_orders"))
    print(f"\nFact_Orders trenutno ima: {result.scalar()} redova")

Duplikat Row ID u staging: 0

Fact_Orders trenutno ima: 0 redova


In [23]:
with engine.connect() as conn:

    # Pronađi redove koji nemaju par u dim_geography
    result = conn.execute(text("""
        SELECT COUNT(*) FROM staging_orders s
        LEFT JOIN dim_geography g 
            ON g.state_or_province = s.`State or Province`
            AND g.city             = s.City
            AND g.postal_code      = s.`Postal Code`
        WHERE g.geography_id IS NULL
    """))
    print(f"Bez Geography: {result.scalar()}")

    # Pronađi redove koji nemaju par u dim_product
    result = conn.execute(text("""
        SELECT COUNT(*) FROM staging_orders s
        LEFT JOIN dim_product p ON p.product_name = s.`Product Name`
        WHERE p.product_key IS NULL
    """))
    print(f"Bez Product: {result.scalar()}")

    # Pronađi redove koji nemaju par u dim_customers
    result = conn.execute(text("""
        SELECT COUNT(*) FROM staging_orders s
        LEFT JOIN dim_customers c ON c.customer_id = s.`Customer ID`
        WHERE c.customer_key IS NULL
    """))
    print(f"Bez Customer: {result.scalar()}")

    # Proveri FK na fact_orders tabeli
    result = conn.execute(text("""
        SELECT COUNT(*) FROM staging_orders s
        LEFT JOIN dim_manager m ON m.manager_name = (
            SELECT Manager FROM staging_users
            WHERE Region = s.Region LIMIT 1
        )
        WHERE m.manager_id IS NULL
    """))
    print(f"Bez Manager: {result.scalar()}")

Bez Geography: 0
Bez Product: 0
Bez Customer: 0
Bez Manager: 0


In [24]:
with engine.connect() as conn:
    # Proveri strukturu fact_orders FK-ova
    result = conn.execute(text("""
        SELECT 
            COLUMN_NAME,
            REFERENCED_TABLE_NAME,
            REFERENCED_COLUMN_NAME
        FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
        WHERE TABLE_NAME = 'fact_orders'
        AND TABLE_SCHEMA = 'smart_store'
        AND REFERENCED_TABLE_NAME IS NOT NULL;
    """))
    
    print("FK veze u fact_orders:")
    print("-" * 50)
    for row in result.fetchall():
        print(f"  {row[0]} → {row[1]}.{row[2]}")

FK veze u fact_orders:
--------------------------------------------------
  customer_id → dim_customers.customer_id
  product_id → dim_product.product_key
  geography_id → dim_geography.geography_id
  order_date_id → dim_calendar.date_id
  ship_date_id → dim_calendar.date_id
  segment_id → dim_segment.segment_id
  orderpriority_id → dim_orderpriority.orderpriority_id
  manager_id → dim_manager.manager_id
  shipmode_id → dim_shipmode.shipmode_id


In [25]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT COLUMN_NAME, REFERENCED_TABLE_NAME, REFERENCED_COLUMN_NAME
        FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
        WHERE TABLE_NAME = 'fact_orders'
        AND TABLE_SCHEMA = 'smart_store'
        AND REFERENCED_TABLE_NAME IS NOT NULL;
    """))
    
    for row in result.fetchall():
        print(f"  {row[0]} → {row[1]}.{row[2]}")

  product_id → dim_product.product_key
  geography_id → dim_geography.geography_id
  order_date_id → dim_calendar.date_id
  ship_date_id → dim_calendar.date_id
  segment_id → dim_segment.segment_id
  orderpriority_id → dim_orderpriority.orderpriority_id
  manager_id → dim_manager.manager_id
  shipmode_id → dim_shipmode.shipmode_id
  customer_id → dim_customers.customer_key


In [26]:
with engine.connect() as conn:
    print("📊 Punjenje Fact_Orders...")
    conn.execute(text("""
        INSERT INTO fact_orders (
            row_id, customer_id, product_id, geography_id,
            order_date_id, ship_date_id, segment_id,
            orderpriority_id, manager_id, shipmode_id,
            order_id, sales, profit, quantity,
            discount, unit_price, shipping_cost, is_returned
        )
        SELECT
            s.`Row ID`,
            c.customer_key,
            p.product_key,
            g.geography_id,
            CAST(DATE_FORMAT(s.`Order Date`, '%Y%m%d') AS UNSIGNED),
            CAST(DATE_FORMAT(s.`Ship Date`,  '%Y%m%d') AS UNSIGNED),
            seg.segment_id,
            op.orderpriority_id,
            m.manager_id,
            sm.shipmode_id,
            s.`Order ID`,
            s.Sales,
            s.Profit,
            s.`Quantity ordered new`,
            s.Discount,
            s.`Unit Price`,
            s.`Shipping Cost`,
            s.is_returned
        FROM staging_orders s
        JOIN dim_customers     c   ON c.customer_id       = s.`Customer ID`
        JOIN dim_product       p   ON p.product_name      = s.`Product Name`
        JOIN dim_geography     g   ON g.state_or_province = s.`State or Province`
                                  AND g.city              = s.City
                                  AND g.postal_code       = s.`Postal Code`
        JOIN dim_segment       seg ON seg.segment_name    = s.`Customer Segment`
        JOIN dim_orderpriority op  ON op.order_priority   = s.`Order Priority`
        JOIN dim_shipmode      sm  ON sm.ship_mode        = s.`Ship Mode`
        JOIN dim_manager       m   ON m.manager_name = (
            SELECT Manager FROM staging_users
            WHERE Region = s.Region
            LIMIT 1
        );
    """))
    conn.commit()
    print(f"✅ Fact_Orders napunjena!")

📊 Punjenje Fact_Orders...
✅ Fact_Orders napunjena!


In [27]:
# Finalna verifikacija
with engine.connect() as conn:
    tables = [
        'fact_orders', 'dim_calendar', 'dim_customers',
        'dim_product', 'dim_geography', 'dim_manager',
        'dim_segment', 'dim_shipmode', 'dim_orderpriority'
    ]
    
    print("📊 Finalni broj redova po tabeli:")
    print("-" * 35)
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"  {table:<25} {count:>6} redova")

📊 Finalni broj redova po tabeli:
-----------------------------------
  fact_orders                 1951 redova
  dim_calendar                 188 redova
  dim_customers               1130 redova
  dim_product                  912 redova
  dim_geography                985 redova
  dim_manager                    4 redova
  dim_segment                    4 redova
  dim_shipmode                   3 redova
  dim_orderpriority              5 redova


In [28]:
# Brzi test - pogledaj 5 redova sa svim dimenzijama
query = """
    SELECT
        f.row_id,
        c.customer_name,
        p.product_name,
        p.product_category,
        g.city,
        g.state_or_province,
        cal.full_date    AS order_date,
        m.manager_name,
        f.sales,
        f.profit,
        f.is_returned
    FROM fact_orders f
    JOIN dim_customers     c   ON c.customer_key  = f.customer_id
    JOIN dim_product       p   ON p.product_key   = f.product_id
    JOIN dim_geography     g   ON g.geography_id  = f.geography_id
    JOIN dim_calendar      cal ON cal.date_id     = f.order_date_id
    JOIN dim_manager       m   ON m.manager_id    = f.manager_id
    LIMIT 5;
"""

pd.read_sql(query, engine)

,row_id,customer_name,product_name,product_category,city,state_or_province,order_date,manager_name,sales,profit,is_returned
0,151,Geoffrey Saunders,GBC Therma-A-Bind 250T Electric Binding System,Office Supplies,New York City,New York,2015-05-05,Erin,5911.35,1408.19,0
1,152,Geoffrey Saunders,Holmes Replacement Filter for HEPA Air Cleaner...,Office Supplies,New York City,New York,2015-05-05,Erin,4649.85,-1069.72,0
2,494,Caroline Johnston,Belkin 105-Key Black Keyboard,Technology,Boston,Massachusetts,2015-06-22,Erin,1233.32,-20.25,0
3,495,Caroline Johnston,Avery Durable Binders,Office Supplies,Boston,Massachusetts,2015-06-22,Erin,47.31,-3.38,0
4,566,Nina Horne Kelly,Euro Pro Shark Stick Mini Vacuum,Office Supplies,Washington,District of Columbia,2015-04-04,Erin,2119.54,-662.52,0
